# Kafka Internals — Replication, Exactly-Once, Consumer Groups, Compaction

## Mental Model

This notebook uses the Citi learning stack running on localhost to demonstrate four Kafka internals that matter in real production systems:

- **ISR (In-Sync Replica) mechanics**: who is fully caught up for each partition, and why `acks=all` depends on ISR health.
- **Exactly-once semantics**: how idempotence and transactions work together so a producer can write a batch atomically.
- **Consumer group rebalancing**: how partitions are assigned, moved, and resumed when consumers join or leave a group.
- **Log compaction**: why key-based topics preserve the latest value per key and eventually remove superseded records.

Citi narrative used throughout this notebook:

- 6,000+ API endpoints monitored for latency, error rate, and throughput
- alerts escalate through severity tiers
- PostgreSQL telemetry is the source of truth for demo records
- Kafka is the event backbone for alerts and endpoint status changes


In [ ]:
import json
import time
from collections import defaultdict

import psycopg2
from confluent_kafka import Consumer, KafkaException, Producer
from confluent_kafka.admin import AdminClient, NewTopic

KAFKA_BOOTSTRAP = 'localhost:9092'
POSTGRES_CONFIG = {
    'host': 'localhost',
    'port': 5432,
    'dbname': 'de_telemetry',
    'user': 'de_admin',
    'password': 'DeAdmin2026!',
}
ALERTS_TOPIC = 'citi.alerts'
STATUS_TOPIC = 'citi.endpoint-status'
admin = AdminClient({'bootstrap.servers': KAFKA_BOOTSTRAP})

def pg_conn():
    return psycopg2.connect(**POSTGRES_CONFIG)

def wait_for_topic(topic_name: str, timeout_s: float = 20.0) -> None:
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        md = admin.list_topics(timeout=10)
        if topic_name in md.topics and not md.topics[topic_name].error:
            return
        time.sleep(0.5)
    raise TimeoutError(f'Topic {topic_name} not visible after creation request')

def ensure_topic(topic_name: str, num_partitions: int, replication_factor: int = 1, config: dict | None = None) -> None:
    md = admin.list_topics(timeout=10)
    if topic_name in md.topics and not md.topics[topic_name].error:
        return
    fs = admin.create_topics([
        NewTopic(topic_name, num_partitions=num_partitions, replication_factor=replication_factor, config=config or {})
    ])
    for _, fut in fs.items():
        try:
            fut.result()
        except Exception as exc:
            if 'already exists' not in str(exc):
                raise
    wait_for_topic(topic_name)

ensure_topic(ALERTS_TOPIC, num_partitions=2, replication_factor=1)
ensure_topic(STATUS_TOPIC, num_partitions=2, replication_factor=1, config={'cleanup.policy': 'compact'})

print(f'Kafka bootstrap: {KAFKA_BOOTSTRAP}')
print(f"PostgreSQL: {POSTGRES_CONFIG['host']}:{POSTGRES_CONFIG['port']}/{POSTGRES_CONFIG['dbname']}")
print(f'Topics ensured: {ALERTS_TOPIC}, {STATUS_TOPIC}')


## ISR Deep Dive

The **ISR (In-Sync Replica)** set is the list of replicas for a partition that are fully caught up enough to be eligible for leader commit guarantees.


In [ ]:
metadata = admin.list_topics(timeout=10)
topic_md = metadata.topics[ALERTS_TOPIC]
print(f'Topic: {ALERTS_TOPIC}')
for partition_id, pmd in sorted(topic_md.partitions.items()):
    print(f'Partition {partition_id} | leader={pmd.leader} | replicas={list(pmd.replicas)} | isr={list(pmd.isrs)}')
print(f'Known brokers: {sorted(metadata.brokers.keys())}')
print('If a broker goes down, ISR can shrink; with acks=all, writes wait on the current ISR set, trading latency for durability.')


## Exactly-Once Semantics

Below, we fetch 10 real alert rows from PostgreSQL and write them to `citi.alerts` inside a single Kafka transaction using `transactional.id = "citi-txn-1"`.


In [ ]:
with pg_conn() as conn:
    with conn.cursor() as cur:
        cur.execute('''
            SELECT a.alert_id, a.endpoint_id, a.severity, a.message, a.created_at,
                   e.name, e.region, e.status, e.category
            FROM alerts a
            JOIN endpoints e ON e.endpoint_id = a.endpoint_id
            ORDER BY a.created_at DESC, a.alert_id DESC
            LIMIT 10
        ''')
        txn_alerts = cur.fetchall()

producer = Producer({
    'bootstrap.servers': KAFKA_BOOTSTRAP,
    'enable.idempotence': True,
    'acks': 'all',
    'transactional.id': 'citi-txn-1',
})
delivered = []

def txn_delivery(err, msg):
    if err is not None:
        raise KafkaException(err)
    delivered.append((msg.topic(), msg.partition(), msg.offset()))

producer.init_transactions()
producer.begin_transaction()
for row in txn_alerts:
    payload = {
        'alert_id': row[0], 'endpoint_id': row[1], 'severity': row[2], 'message': row[3],
        'created_at': row[4].isoformat(), 'endpoint_name': row[5], 'region': row[6],
        'endpoint_status': row[7], 'category': row[8],
    }
    producer.produce(ALERTS_TOPIC, key=str(row[1]).encode('utf-8'), value=json.dumps(payload).encode('utf-8'), callback=txn_delivery)
producer.commit_transaction()
producer.flush(10)
print('Transaction committed — 10 alerts written exactly once')
print(f'Delivery reports captured: {len(delivered)}')


## Consumer Group Mechanics


In [ ]:
with pg_conn() as conn:
    with conn.cursor() as cur:
        cur.execute('''
            SELECT a.alert_id, a.endpoint_id, a.severity, a.message, a.created_at, e.name, e.region
            FROM alerts a
            JOIN endpoints e ON e.endpoint_id = a.endpoint_id
            ORDER BY a.created_at DESC, a.alert_id DESC
            LIMIT 20
        ''')
        group_alerts = cur.fetchall()

seed_producer = Producer({'bootstrap.servers': KAFKA_BOOTSTRAP, 'enable.idempotence': True, 'acks': 'all'})
for row in group_alerts:
    payload = {
        'demo_type': 'consumer_group_seed', 'alert_id': row[0], 'endpoint_id': row[1], 'severity': row[2],
        'message': row[3], 'created_at': row[4].isoformat(), 'endpoint_name': row[5], 'region': row[6],
    }
    seed_producer.produce(ALERTS_TOPIC, key=str(row[1]).encode('utf-8'), value=json.dumps(payload).encode('utf-8'))
seed_producer.flush(10)

assignments = {'consumer_1': [], 'consumer_2': []}

def make_on_assign(name):
    def _on_assign(consumer, partitions):
        assignments[name] = [f'{p.topic}[{p.partition}]' for p in partitions]
        consumer.assign(partitions)
        print(f'{name} assigned -> {assignments[name]}')
    return _on_assign

def make_on_revoke(name):
    def _on_revoke(consumer, partitions):
        consumer.unassign()
        print(f"{name} revoked -> {[f'{p.topic}[{p.partition}]' for p in partitions]}")
    return _on_revoke

common_cfg = {
    'bootstrap.servers': KAFKA_BOOTSTRAP,
    'group.id': 'citi-deep-group',
    'auto.offset.reset': 'latest',
    'enable.auto.commit': False,
    'session.timeout.ms': 6000,
}
consumer_1 = Consumer(common_cfg)
consumer_2 = Consumer(common_cfg)
consumer_1.subscribe([ALERTS_TOPIC], on_assign=make_on_assign('consumer_1'), on_revoke=make_on_revoke('consumer_1'))
consumer_2.subscribe([ALERTS_TOPIC], on_assign=make_on_assign('consumer_2'), on_revoke=make_on_revoke('consumer_2'))

join_deadline = time.time() + 15
while time.time() < join_deadline:
    consumer_1.poll(0.2)
    consumer_2.poll(0.2)
    if assignments['consumer_1'] and assignments['consumer_2']:
        break

counts = {'consumer_1': 0, 'consumer_2': 0}
total = 0
deadline = time.time() + 20
while total < 20 and time.time() < deadline:
    msg1 = consumer_1.poll(0.5)
    if msg1 is not None and not msg1.error():
        counts['consumer_1'] += 1
        total += 1
    if total >= 20:
        break
    msg2 = consumer_2.poll(0.5)
    if msg2 is not None and not msg2.error():
        counts['consumer_2'] += 1
        total += 1

print(f"Consumer 1 assignments: {assignments['consumer_1']}")
print(f"Consumer 2 assignments: {assignments['consumer_2']}")
print(f"Messages consumed by consumer_1: {counts['consumer_1']}")
print(f"Messages consumed by consumer_2: {counts['consumer_2']}")

consumer_2.close()
print('consumer_2 closed — waiting for rebalance on consumer_1')

rebalance_seed = Producer({'bootstrap.servers': KAFKA_BOOTSTRAP, 'enable.idempotence': True, 'acks': 'all'})
for i in range(5):
    rebalance_seed.produce(ALERTS_TOPIC, key=f'rebalance-{i}'.encode('utf-8'), value=json.dumps({'demo_type': 'rebalance_after_consumer_exit', 'sequence': i}).encode('utf-8'))
rebalance_seed.flush(10)

rebalance_deadline = time.time() + 15
while time.time() < rebalance_deadline:
    consumer_1.poll(0.5)
    if assignments['consumer_1']:
        break
print(f"Post-exit consumer_1 assignments: {assignments['consumer_1']}")
consumer_1.close()


## Log Compaction


In [ ]:
with pg_conn() as conn:
    with conn.cursor() as cur:
        cur.execute('''
            SELECT endpoint_id, name, region, status, category
            FROM endpoints
            ORDER BY endpoint_id
            LIMIT 5
        ''')
        endpoints = cur.fetchall()

status_cycle = ['HEALTHY', 'DEGRADED', 'RECOVERING', 'HEALTHY']
status_producer = Producer({'bootstrap.servers': KAFKA_BOOTSTRAP, 'enable.idempotence': True, 'acks': 'all'})
events_written = 0
for endpoint_id, name, region, current_status, category in endpoints:
    for idx, new_status in enumerate(status_cycle):
        value = {
            'endpoint_id': endpoint_id, 'name': name, 'region': region, 'category': category,
            'status': new_status, 'event_index': idx, 'source_status': current_status,
            'emitted_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
        }
        status_producer.produce(STATUS_TOPIC, key=str(endpoint_id).encode('utf-8'), value=json.dumps(value).encode('utf-8'))
        events_written += 1

tombstone_endpoint_id = endpoints[0][0]
status_producer.produce(STATUS_TOPIC, key=str(tombstone_endpoint_id).encode('utf-8'), value=None)
status_producer.flush(10)
print(f'Status events written before tombstone: {events_written}')
print(f'Tombstone written for endpoint_id={tombstone_endpoint_id}')
print('Compacted topic created — latest value per key preserved')


## What Just Happened

- **ISR**: inspected leaders, replicas, and ISR membership for `citi.alerts`
- **Exactly-once semantics**: wrote 10 alert events in a single committed transaction
- **Consumer groups**: split partitions across two consumers, then triggered a rebalance when one left
- **Compaction**: built a key-based endpoint-status topic and wrote a tombstone delete marker

**Citi interview angle**: ISR shrink + `acks=all` = the tradeoff Citi makes between latency and durability.
